### Esercizi SQL Esteso

Sono date le seguenti tabelle:

**NEGOZIO** (*IDNegozio*, Negozio, Città, Regione, AreaGeografica)

**MODELLO-DISP-ELETTRONICO** (*IDModDispEle*, Modello, Categoria) 
    
**JUNK-CARAT-COPERTURE-ASSICURATIVE** (*IDJCCA*, GaranziaLegale, Estensione3Anni, DanniAccidentali, Furto)

**JUNK-CARAT-ACQUIRENTE** (*IDJCA*, Genere, Residenza, FasciaEtà)

**TEMPO** (*IDTempo*, Data, Mese, 2-Mesi, 3-Mesi, 6-Mesi, Anno)

**RICHIESTE-RIMBORSO** (*IDNegozio*, IDModDispEle, IDJCCA, IDJCA, IDTempo, #RichiesteRicevute, #RichiesteConcluse, ImportoTotRichiesto, ImportoTotApprovato, DurataTotProcesso)

1. Visualizzare separatamente per negozio, categoria del dispositivo elettronico e bimestre (attributo 2-mesi):

    •⁠  ⁠La percentuale dell'importo richiesto che è stato approvato

    •⁠  ⁠La differenza tra il numero di richieste ricevute e quelle concluse

    •⁠  ⁠L'importo complessivo approvato indipendentemente dal negozio

    •⁠  ⁠Il rapporto tra l'importo approvato e l'importo complessivo di tutti i negozi siti nella stessa città, separatamente per categoria e bimestre

In [ ]:
SELECT N.Negozio, M.Categoria, T."2-Mesi",
        -- 1. Percentuale importo richiesto approvato
        100 * SUM(RR.ImportoTotApprovato) / SUM (RR.ImportoTotRichiesto) AS PercImportoApprovato,
        -- 2. Differenza richieste ricevute e concluse
        SUM(RR."#RichiesteRicevute") - SUM(RR."#RichiesteConcluse") AS Richiete-Ricevute-Approvate,
        -- 3. Importo complessivo approvato indipendentemente dal negozio
        SUM(SUM(RR.ImportoTotApprovato)) OVER (
            PARTITION BY M.Categoria, T."2-Mesi"
        ) AS ImportoTotApprovato,
        -- 4. Rapporto rispetto a tutti i negozi della stessa città (per categoria e bimestre)
        SUM(RR.ImportoTotApprovato) / SUM(SUM(RR.ImportoTotApprovato)) OVER (
            PARTITION BY N.Città, M.Categoria, T."2-Mesi"
        ) AS RapportoSuCittà
FROM "RICHIESTE-RIMBORSO" RR 
JOIN NEGOZIO N ON N.IDNegozio = RR.IDNegozio
JOIN "MODELLO-DISP-ELETTRONICO" M ON M.IDModDisEle = RR.IDModDisEle
JOIN TEMPO T ON T.IDTempo = RR.IDTempo
GROUP BY N.Negozio, N.Città, M.Categoria, T."2-Mesi"

2. Considerando le coperture assicurative che includono solo la garanzia legale (attributo GaranziaLegale), separatamente per trimestre e città, visualizzare:

- ⁠La percentuale di richieste concluse rispetto a quelle ricevute

- ⁠La durata media del tempo di processamento per richiesta ricevuta

    Associare ad ogni record visualizzato la posizione in un ranking:

- ⁠in funzione dell'importo complessivo approvato separatamente per area geografica (1 per il record con il più basso valore complessivo approvato)

- in funzione della differenza tra importo richiesto e importo approvato (1 per il record con il più alto valore della differenza tra importo richiesto e importo approvato complessivi) separatamente per anno

In [ ]:
SELECT T."3-Mesi", N.Città,
        100 * SUM(RR."#RichiesteConcluse") / SUM(RR."#RichiesteRicevute"),
        SUM(RR.DurataTotProcesso) / SUM(RR."#RichiesteConcluse"),
        RANK() OVER(
            PARTITION BY N.AreaGeografica
            ORDER BY SUM(RR.ImportoTotApprovato) ASC 
        )
        RANK() OVER(
            PARTITION BY T.Anno
            ORDER BY SUM(RR.ImportoTotRichiesto) - SUM(RR.ImportoTotApprovato) DESC
        )
FROM "RICHIESTE-RIMBORSO" RR 
JOIN "JUNK-CARAT-COPERTURE-ASSICURATIVE" J ON J.IDJCCA = RR.IDJCCA
JOIN NEGOZIO N ON N.IDNegozio = RR.IDNegozio
JOIN TEMPO T ON T.IDTempo = RR.IDTempo
WHERE J.GaranziaLegale = 1 
    AND J.Estensione3Anni = 0
    AND J.DanniAccidentali = 0
    AND J.Furto = 0
GROUP BY T."3-Mesi", N.Città, N.AreaGeografica, T.Anno

3. Considerando gli anni precedenti al 2020, separatamente per modello del dispositivo elettronico, semestre (attributo 6-Mesi) e genere dell'acquirente, visualizzare:

- ⁠L'importo approvato medio per richiesta conclusa

- ⁠L'importo cumulativo approvato al trascorrere dei semestri, separatamente per modello del dispositivo elettronico

- L'importo complessivo richiesto indipendentemente dal modello del dispositivo, semestre e Genere

In [ ]:
SELECT M.Modello, T."6-Mesi", J.Genere,
        SUM(RR.ImportoTotApprovato) / SUM(RR."#RichiesteConcluse"),
        SUM(SUM(RR.ImportoTotApprovato)) OVER (
            PARTITION BY M.Modello
            ORDER BY T."6-Mesi" 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENTT ROW 
        )
        SUM(SUM(RR.ImportoTotRichiesto)) OVER ()
FROM "RICHIESTE-RIMBORSO" RR 
JOIN TEMPO T ON T.IDTempo = RR.IDTempo
JOIN "JUNK-CARAT-ACQUIRENTE" J ON J.IDJCA = RR.IDJCA
JOIN "MODELLO-DISP-ELETTRONICO" M ON M.IDModDisEle = RR.IDModDisEle
WHERE T.Anno < 2020
GROUP BY M.Modello, T."6-Mesi", J.Genere

Dato lo schema logico precedente, considerare le seguenti query di interesse:

1. Considerando l'area geografica nord, separatamente per regione e anno, visualizzare il numero complessivo di richieste concluse, l'importo complessivo approvato e la durata media mensile di processamento delle richieste.

2. Considerando le coperture assicurative che includono i danni accidentali (attributo DanniAccidentali), ma non il furto (attributo Furto), separatamente per mese e area geografica, visualizzare il numero complessivo di richieste concluse e la differenza tra l'importo complessivo richiesto e quello approvato.

3. Considerando gli anni 2021 e 2022, separatamente per semestre (attributo 6-Mesi) e regione, visualizzare il numero complessivo di richieste concluse e il corrispondente importo medio approvato.

1) ⁠Definire una vista materializzata con CREATE MATERIALIZED VIEW, in modo da ridurre il tempo di risposta delle query di interesse da (1) a (3) sopra riportate.

In [ ]:
CREATE MATERIALIZED VIEW ViewRimborsi
BUILD IMMEDIATE 
REFRESH FAST ON COMMIT
AS
SELECT N.Regione, N.AreaGeografica, T.Mese, T."6-Mesi", T.Anno, J.DanniAccidentali, J.Furto
        SUM(RR."#RichiesteConcluse") AS RichiesteConcluse,
        SUM(RR.ImportoTotApprovato) AS ImportoTotApprovato,
        SUM(RR.DurataTotProcesso) AS DurataTotProcesso,
        SUM(RR.ImportoTotRichiesto) AS ImportoTotRichiesto
FROM "RICHIESTE-RIMBORSO" RR 
JOIN NEGOZIO N ON N.IDNegozio = RR.IDNegozio
JOIN TEMPO T ON T.IDTempo = RR.IDTempo
JOIN "JUNK-CARAT-COPERTURE-ASSICURATIVE" J ON J.IDJCCA = RR.IDJCCA
GROUP BY N.Regione, N.AreaGeografica, T.Mese, T."6-Mesi", T.Anno, J.DanniAccidentali, J.Furto

2) ⁠Definire l'insieme minimale di attributi che permette di identificare le tuple appartenenti alla vista materializzata ViewRimborsi.

    Insieme Minimale : {**Regione, Mese, DanniAccidentali, Furto**}

3. ⁠Si ipotizzi che la gestione della vista materializzata (tabella derivata) sia svolta mediante trigger. Scrivere il trigger per propagare le modifiche alla vista materializzata ViewRimborsi in caso di inserimento di un nuovo record nella tabella dei fatti RICHIESTE-RIMBORSO. 

In [ ]:
CREATE OR REPLACE TRIGGER TrigRichiesteRimborso
AFTER INSERT ON "RICHIESTE-RIMBORSO"
FOR EACH ROW

DECLARE
VarRegione, VarAreaGeografica VARCHAR(20);
VarMese, VarSemestre, VarAnno DATE;
VarDanniAccidentali, VarFurto BOOLEAN;
N INTEGER;

BEGIN

    SELECT Mese, "6-Mesi", Anno INTO VarMese, VarSemestre, VarAnno
    FROM TEMPO
    WHERE IDTempo = :NEW.IDTempo;

    SELECT DanniAccidentali, Furto INTO VarDanniAccidentali, VarFurto
    FROM "JUNK-CARAT-COPERTURE-ASSICURATIVE"
    WHERE IDJCCA = :NEW.IDJCCA;

    SELECT Regione, AreaGEografica INTO VarRegione, VarAreaGeografica
    FROM NEGOZIO
    WHERE IDNegozio = :NEW.IDNegozio;

    SELECT COUNT(*) INTO N 
    FROM ViewRimborsi 
    WHERE Mese = VarMese
        AND Regione = VarRegione
        AND DanniAccidentali = VarDanniAccidentali
        AND Furto = VarFurto;

    IF N > 0 THEN
        UPDATE ViewRimborsi
        SET RichiesteConcluse = RichiesteConcluse + :NEW."#RichiesteConcluse",
            ImportoTotApprovato = ImportoTotApprovato + :NEW.ImportoTotApprovato,
            DurataTotProcesso = DurataTotProcesso + :NEW.DurataTotProcesso,
            ImportoRichiesto = ImportoRichiesto + :NEW.ImportoTotRichiesto
        WHERE Mese = VarMese
            AND Regione = VarRegione
            AND DanniAccidentali = VarDanniAccidentali
            AND Furto = VarFurto;
    ELSE 
        INSERT INTO ViewRimborsi(Regione, AreaGeografica, Mese, "6-Mesi", Anno, DanniAccidentali, Furto,
            RichiesteConcluse, ImportoTotApprovato, DurataTotProcesso, ImportoTotRichiesto)
        VALUES(
            VarRegione, VarAreaGeografica, VarMese, VarSemestre, VarAnno, VarDanniAccidentali, VarFurto,
            :NEW."#RichiesteConcluse", :NEW.ImportoTotApprovato, :NEW.DurataTotProcesso, :NEW.ImportoTotRichiesto
        );
        END IF;
    END;
